In [ ]:
# Week 7 — LLM Foundations and Prompt Engineering

## Day 1 — How LLM Applications Work

### Objective

Understand the core concepts behind LLM-powered applications through
controlled experiments:

- Tokens
- Context windows
- Embeddings
- Attention at a conceptual level
- Sampling
- Latency
- Cost
- Hosted vs. local models
- Privacy and licensing considerations

The experiments in this notebook are designed to understand LLM
application behavior rather than train an LLM from scratch.

In [1]:
import os
import time
import json
from pathlib import Path

print("Python environment ready.")

Python environment ready.


In [2]:
import sys

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

Python executable:
c:\Users\raazi\Documents\intern-project\.venv\Scripts\python.exe

Python version:
3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]


In [3]:
import openai

print("OpenAI SDK version:", openai.__version__)

OpenAI SDK version: 3.8.0


In [4]:
import os

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("OPENAI_API_KEY is available.")
    print("Key length:", len(api_key))
else:
    print("OPENAI_API_KEY is not available.")

OPENAI_API_KEY is not available.


In [5]:
from dotenv import load_dotenv

load_dotenv()

print("Environment file loaded.")

Environment file loaded.


In [7]:
import os

api_key = os.getenv("OPENAI_API_KEY")

print("API key available:", bool(api_key))

API key available: False


## API Configuration

The project supports hosted LLM experimentation through the OpenAI SDK.
No API key is currently configured in the local `.env` file.

Therefore, this notebook separates:
1. Local, reproducible LLM concepts and experiments
2. API-dependent experiments

No API credentials are stored directly in the notebook.


In [9]:
import os

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("OpenAI API configuration: available")
else:
    print("OpenAI API configuration: not configured")

OpenAI API configuration: not configured


## Experiment 1 — Tokens

LLMs do not process text directly as complete words or sentences.

Text is first converted into smaller units called **tokens**. Depending on
the tokenizer and language, a token may represent a whole word, part of a
word, punctuation, whitespace, or another text fragment.

Tokenization matters because token counts affect:

- How much text can fit into a model's context window
- API usage and cost
- Processing requirements
- The amount of information an LLM can receive in a single request

In [10]:
import tiktoken

# GPT-style tokenizer
encoding = tiktoken.get_encoding("cl100k_base")

print("Tokenizer loaded:", encoding.name)

Tokenizer loaded: cl100k_base


In [11]:
texts = [
    "Hello world!",
    "Machine learning is useful.",
    "The customer purchased a laptop.",
    "This is a longer sentence showing how tokenization works in practice.",
]

for text in texts:
    tokens = encoding.encode(text)

    print(f"Text: {text}")
    print(f"Token IDs: {tokens}")
    print(f"Token count: {len(tokens)}")
    print("-" * 60)

Text: Hello world!
Token IDs: [9906, 1917, 0]
Token count: 3
------------------------------------------------------------
Text: Machine learning is useful.
Token IDs: [22333, 6975, 374, 5505, 13]
Token count: 5
------------------------------------------------------------
Text: The customer purchased a laptop.
Token IDs: [791, 6130, 15075, 264, 21288, 13]
Token count: 6
------------------------------------------------------------
Text: This is a longer sentence showing how tokenization works in practice.
Token IDs: [2028, 374, 264, 5129, 11914, 9204, 1268, 4037, 2065, 4375, 304, 6725, 13]
Token count: 13
------------------------------------------------------------


In [12]:
def analyze_text(text):
    tokens = encoding.encode(text)

    return {
        "characters": len(text),
        "words": len(text.split()),
        "tokens": len(tokens),
        "characters_per_token": round(len(text) / len(tokens), 2)
    }


examples = [
    "Hello world!",
    "Data science",
    "Customer retention analysis",
    "Machine learning and artificial intelligence",
]

for text in examples:
    print(text)
    print(analyze_text(text))
    print("-" * 60)

Hello world!
{'characters': 12, 'words': 2, 'tokens': 3, 'characters_per_token': 4.0}
------------------------------------------------------------
Data science
{'characters': 12, 'words': 2, 'tokens': 2, 'characters_per_token': 6.0}
------------------------------------------------------------
Customer retention analysis
{'characters': 27, 'words': 3, 'tokens': 3, 'characters_per_token': 9.0}
------------------------------------------------------------
Machine learning and artificial intelligence
{'characters': 44, 'words': 5, 'tokens': 5, 'characters_per_token': 8.8}
------------------------------------------------------------


### Observations

The experiment demonstrates that token count is not equivalent to word
count or character count.

For the tested examples:

- `Hello world!` contains 2 words but 3 tokens.
- `Data science` contains 2 words and 2 tokens.
- `Customer retention analysis` contains 3 words and 3 tokens.
- `Machine learning and artificial intelligence` contains 5 words and
  5 tokens.

The measured character-to-token ratio ranged from 4.0 to 9.0 in these
examples.

Therefore, character count and word count should not be treated as exact
proxies for token count. Tokenization depends on the tokenizer and the
specific text being processed.

### Engineering relevance

For an LLM application, token count is important because it affects:

- Context usage
- Input/output limits
- API usage and cost
- Processing requirements
- Prompt efficiency

## Experiment 2 — Tokenization Depends on the Text

Different types of text can produce very different tokenization behavior.

We will compare:

- Normal English text
- Repeated text
- Numbers
- Punctuation
- Code
- Long words

In [13]:
test_cases = {
    "Normal English": "The customer bought a new phone.",
    "Repeated text": "customer customer customer customer customer",
    "Numbers": "1234567890 99.99 2026",
    "Punctuation": "Hello!!! What??? Really...",
    "Code": "SELECT customer_id, total_amount FROM orders;",
    "Long word": "electromagnetically"
}

for name, text in test_cases.items():
    tokens = encoding.encode(text)

    print(f"{name}:")
    print(f"Text: {text}")
    print(f"Token count: {len(tokens)}")
    print(f"Tokens: {tokens}")
    print("-" * 70)
    

Normal English:
Text: The customer bought a new phone.
Token count: 7
Tokens: [791, 6130, 11021, 264, 502, 4641, 13]
----------------------------------------------------------------------
Repeated text:
Text: customer customer customer customer customer
Token count: 5
Tokens: [11296, 6130, 6130, 6130, 6130]
----------------------------------------------------------------------
Numbers:
Text: 1234567890 99.99 2026
Token count: 11
Tokens: [4513, 10961, 16474, 15, 220, 1484, 13, 1484, 220, 2366, 21]
----------------------------------------------------------------------
Punctuation:
Text: Hello!!! What??? Really...
Token count: 6
Tokens: [9906, 12340, 3639, 34115, 29308, 1131]
----------------------------------------------------------------------
Code:
Text: SELECT customer_id, total_amount FROM orders;
Token count: 9
Tokens: [4963, 6130, 851, 11, 2860, 13779, 4393, 10373, 26]
----------------------------------------------------------------------
Long word:
Text: electromagnetically
Token 

### Observations

The second experiment shows that tokenization behavior depends strongly on
the input text.

Key observations:

1. A single word can consist of multiple tokens.
   - `electromagnetically` → 4 tokens

2. Numbers may be divided into multiple tokens.
   - `1234567890 99.99 2026` → 11 tokens

3. Punctuation contributes to tokenization.
   - `Hello!!! What??? Really...` → 6 tokens

4. Code is also represented as tokens.
   - The SQL example produced 9 tokens.

5. Repeated words do not necessarily produce the same token sequence as
   a single isolated occurrence because whitespace and token boundaries
   affect tokenization.

These results reinforce that word count is not an exact substitute for
token count.

### Engineering relevance

An LLM application should estimate and monitor token usage rather than
assuming that the number of words directly represents the amount of
information being sent to the model.

In [14]:
text = "The customer purchased a laptop."

tokens = encoding.encode(text)

print("Original text:")
print(text)

print("\nToken pieces:")

for token in tokens:
    piece = encoding.decode([token])
    print(f"{token:>6} -> {repr(piece)}")

Original text:
The customer purchased a laptop.

Token pieces:
   791 -> 'The'
  6130 -> ' customer'
 15075 -> ' purchased'
   264 -> ' a'
 21288 -> ' laptop'
    13 -> '.'


### Experiment 3 — Inspecting Token Pieces

Instead of examining only token IDs, the tokenizer output was decoded
back into the corresponding token pieces.

For the sentence:

`The customer purchased a laptop.`

the tokenizer produced:

- `The`
- ` customer`
- ` purchased`
- ` a`
- ` laptop`
- `.`

This demonstrates that token boundaries do not necessarily correspond
directly to whitespace-separated words.

Some token pieces contain a leading space, while punctuation can be
represented as its own token.

### Key takeaway

The LLM receives a sequence of tokenizer-generated tokens rather than
ordinary words. Therefore, token count depends on the tokenizer and the
exact input text.

## Experiment 4 — Tokens and Context Windows

An LLM has a finite context window: the maximum amount of tokenized
information that can be processed within a request.

The context can include:

- System instructions
- User prompts
- Conversation history
- Retrieved information
- Other input provided to the model
- Model output, depending on the model and API

Therefore, token usage is an important engineering constraint.

A longer prompt consumes more of the available context than a shorter
prompt.

In [15]:
short_text = "Customer purchased a laptop."

long_text = short_text * 20

short_tokens = len(encoding.encode(short_text))
long_tokens = len(encoding.encode(long_text))

print("Short text:")
print(f"Characters: {len(short_text)}")
print(f"Tokens: {short_tokens}")

print("\nRepeated text:")
print(f"Characters: {len(long_text)}")
print(f"Tokens: {long_tokens}")

print("\nToken increase:")
print(long_tokens / short_tokens)

Short text:
Characters: 28
Tokens: 5

Repeated text:
Characters: 560
Tokens: 81

Token increase:
16.2


### Observations

The short example contained:

- 28 characters
- 5 tokens

After repeating the same sentence 20 times:

- 560 characters
- 81 tokens

The character count increased by exactly 20×, while the token count
increased by approximately 16.2×.

This demonstrates that token count does not scale through a simple
character-to-token conversion. The tokenizer determines how the input is
segmented.

### Engineering relevance

As an LLM application's input grows, more tokens are consumed from the
available context.

For the Customer Insights Assistant, the context could include system
instructions, customer information, retrieved business information,
conversation history, and the user's question.

Therefore, controlling input length is an important part of LLM
application design.

## Experiment 5 — Context Window Simulation

A context window represents the amount of tokenized information that a
model can process within a request.

For this experiment, we will simulate a small context limit.

Instead of using a real model's context limit, we will define an
artificial limit of 50 tokens and observe what happens as input grows.

In [16]:
CONTEXT_LIMIT = 50

texts = [
    "Customer purchased a laptop.",
    "Customer purchased a laptop. " * 5,
    "Customer purchased a laptop. " * 10,
    "Customer purchased a laptop. " * 20,
]

for text in texts:
    token_count = len(encoding.encode(text))

    status = "Fits" if token_count <= CONTEXT_LIMIT else "Exceeds limit"

    print(f"Tokens: {token_count:>3} | Status: {status}")

Tokens:   5 | Status: Fits
Tokens:  26 | Status: Fits
Tokens:  51 | Status: Exceeds limit
Tokens: 101 | Status: Exceeds limit


### Observations

Using an artificial context limit of 50 tokens:

| Repetitions | Token Count | Result |
|------------:|------------:|--------|
| 1 | 5 | Fits |
| 5 | 26 | Fits |
| 10 | 51 | Exceeds limit |
| 20 | 101 | Exceeds limit |

The experiment demonstrates that an input can cross a context boundary as
its token count increases.

The 10-repetition example required 51 tokens and therefore exceeded the
50-token simulated limit.

This is a simplified simulation rather than an actual model context
limit. Its purpose is to demonstrate the relationship between tokenized
input size and available context.

### Engineering relevance

In a real LLM application, context may be consumed by multiple
components, including instructions, user input, conversation history,
retrieved information, and other model input.

Therefore, an application needs to manage context deliberately rather
than continuously adding information.

## What Can Occupy an LLM Context?

An LLM request can contain multiple types of information.

For an LLM-powered application, relevant context can include:

1. System instructions
2. User input
3. Conversation history
4. Retrieved information
5. Structured data
6. Examples included in a prompt
7. Other application-provided context

All of this contributes to the information the model must process.

This makes context management an important part of LLM application
engineering.

## Experiment 6 — Embeddings

An embedding represents information as a numerical vector.

Instead of representing text only as a sequence of tokens, an embedding
maps the text into a vector space where texts with related meanings can
have similar representations.

Conceptually:

Text
↓
Embedding model
↓
Numerical vector
↓
[0.12, -0.43, 0.87, ...]

Embeddings are important for applications such as:

- Semantic similarity
- Search
- Recommendation systems
- Document retrieval
- Retrieval-Augmented Generation (RAG)

This experiment uses a local, lightweight representation to demonstrate
the idea of vector similarity without requiring an external API.

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

print("TF-IDF vectorizer ready.")

TF-IDF vectorizer ready.


In [18]:
documents = [
    "The customer bought a laptop",
    "The customer purchased a computer",
    "The customer ordered a phone",
    "The weather is sunny today"
]

vectorizer = TfidfVectorizer()
vectors = vectorizer.fit_transform(documents)

print("Number of documents:", vectors.shape[0])
print("Number of features:", vectors.shape[1])

Number of documents: 4
Number of features: 12


In [19]:
feature_names = vectorizer.get_feature_names_out()

print("Features:")
print(feature_names)

print("\nVector for first document:")
print(vectors[0].toarray())

Features:
['bought' 'computer' 'customer' 'is' 'laptop' 'ordered' 'phone'
 'purchased' 'sunny' 'the' 'today' 'weather']

Vector for first document:
[[0.61087812 0.         0.38991559 0.         0.61087812 0.
  0.         0.         0.         0.31878155 0.         0.        ]]


In [20]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(vectors)

print("Cosine similarity matrix:")
print(similarity_matrix)

Cosine similarity matrix:
[[1.         0.25365584 0.25365584 0.08048222]
 [0.25365584 1.         0.25365584 0.08048222]
 [0.25365584 0.25365584 1.         0.08048222]
 [0.08048222 0.08048222 0.08048222 1.        ]]


In [21]:
for i in range(len(documents)):
    for j in range(i + 1, len(documents)):
        print(
            f"Document {i + 1} ↔ Document {j + 1}: "
            f"{similarity_matrix[i, j]:.3f}"
        )

Document 1 ↔ Document 2: 0.254
Document 1 ↔ Document 3: 0.254
Document 1 ↔ Document 4: 0.080
Document 2 ↔ Document 3: 0.254
Document 2 ↔ Document 4: 0.080
Document 3 ↔ Document 4: 0.080


### Observations

TF-IDF vectors were compared using cosine similarity.

The results showed:

- Documents 1–3 had a similarity of approximately 0.254.
- Document 4 had a lower similarity of approximately 0.080 with the
  other documents.

Documents 1–3 share vocabulary such as `customer` and `the`, while
Document 4 is about weather and has less vocabulary overlap.

However, TF-IDF is primarily based on lexical/statistical information.
It does not reliably capture semantic relationships between different
words.

For example:

- `bought` and `purchased`
- `laptop` and `computer`

may be semantically related even though they are different terms.

### Important distinction

TF-IDF is being used in this notebook as a simple demonstration of
representing text as vectors and comparing those vectors.

It is **not a neural embedding model**.

Modern embedding models generate dense numerical representations that
are designed to capture semantic relationships more effectively.

This distinction will become important when implementing embeddings,
vector search, and RAG in Week 8.

In [22]:
semantic_examples = [
    "The customer purchased a laptop",
    "The customer bought a computer",
    "The customer purchased a laptop",
]

semantic_vectors = vectorizer.fit_transform(semantic_examples)

semantic_similarity = cosine_similarity(semantic_vectors)

for i in range(len(semantic_examples)):
    for j in range(i + 1, len(semantic_examples)):
        print(
            f"Sentence {i + 1} ↔ Sentence {j + 1}: "
            f"{semantic_similarity[i, j]:.3f}"
        )

Sentence 1 ↔ Sentence 2: 0.312
Sentence 1 ↔ Sentence 3: 1.000
Sentence 2 ↔ Sentence 3: 0.312


### Semantic Similarity Limitation

A second experiment compared closely related sentences:

1. `The customer purchased a laptop`
2. `The customer bought a computer`
3. `The customer purchased a laptop`

The results were:

- Sentence 1 ↔ Sentence 2: 0.312
- Sentence 1 ↔ Sentence 3: 1.000
- Sentence 2 ↔ Sentence 3: 0.312

The identical sentences received a similarity of 1.000.

The first and second sentences are semantically related, but their TF-IDF
similarity was only 0.312 because several important words differ:

- `purchased` vs. `bought`
- `laptop` vs. `computer`

This illustrates a limitation of lexical vectorization: related meanings
can be expressed using different vocabulary.

Modern neural embedding models are designed to represent semantic
relationships more effectively. Actual embedding models and vector
retrieval will be implemented in Week 8.

## Experiment 7 — Attention at a Conceptual Level

Attention allows a model to determine which parts of the input are
important in relation to the current token or representation.

For example:

`The customer returned the laptop because it was defective.`

To interpret `it`, the model needs to consider the surrounding context,
particularly the relationship between `it` and `laptop`.

Conceptually, attention allows information from different positions in
the input to influence the representation being processed.

A simplified view is:

Input tokens
    ↓
Relationships between tokens
    ↓
Attention weights
    ↓
Context-aware representations

The model can therefore give different levels of attention to different
parts of the input depending on the task and context.

### Key Takeaway

Attention is a mechanism that helps Transformer-based models capture
relationships between tokens within the input context.

It allows the representation of a token to be influenced by relevant
information elsewhere in the context.

For this internship, the goal is conceptual understanding rather than
implementing the attention mechanism from scratch.

## Experiment 8 — Sampling

LLMs generate output by selecting tokens from a probability distribution.

Sampling parameters influence how deterministic or varied the generated
output can be.

A lower temperature generally makes generation more focused and
predictable, while a higher temperature generally allows more variation.

For reproducible application behavior, deterministic or low-variation
settings can be useful. More variation can be useful for creative
generation.

The exact behavior depends on the model and generation configuration.

## Experiment 8 — Sampling

LLMs generate output by selecting tokens from a probability distribution.

Sampling parameters influence how deterministic or varied the generated
output can be.

A lower temperature generally makes generation more focused and
predictable, while a higher temperature generally allows more variation.

For reproducible application behavior, deterministic or low-variation
settings can be useful. More variation can be useful for creative
generation.

The exact behavior depends on the model and generation configuration.

## Hosted vs Local Models

### Hosted models

A hosted model runs on infrastructure provided by an external service.

Advantages can include:

- Easy access through an API
- No requirement to operate model infrastructure locally
- Access to powerful models

Considerations include:

- API dependency
- Usage cost
- Data handling and privacy requirements
- Network latency
- Provider-specific licensing and terms

### Local models

A local model runs on infrastructure controlled by the application
developer or organization.

Potential advantages include:

- Greater control over data processing
- Reduced dependency on external APIs
- Ability to operate in environments with restricted connectivity

Considerations include:

- Hardware requirements
- Model size
- Deployment complexity
- Inference performance
- Maintenance

The internship curriculum allows both hosted APIs and local models. The
focus is on sound application engineering rather than dependence on a
specific vendor or framework.

## Privacy and Licensing Considerations

Before sending application data to a hosted LLM, an application should
consider:

- What data is being transmitted
- Whether personal or confidential information is included
- How the provider handles submitted data
- Applicable organizational policies
- Applicable legal or regulatory requirements

Model licensing should also be reviewed before deploying a model,
especially when using local or open-weight models.

Important questions include:

- Is commercial use permitted?
- What attribution requirements exist?
- Are there usage restrictions?
- What additional terms apply to the model or its outputs?

These considerations are part of responsible LLM application engineering.

# Day 1 Summary

The experiments established the main foundations required for LLM
application development.

### Concepts covered

- Tokens and tokenization
- Context windows
- Vector representations
- Similarity
- Limitations of lexical representations
- Attention at a conceptual level
- Sampling
- Latency
- Cost
- Hosted vs local models
- Privacy and licensing

### Main engineering lessons

1. LLMs process tokenized representations rather than ordinary words.
2. Token count affects context usage and potentially cost.
3. Context must be managed deliberately.
4. Text can be represented numerically and compared using vectors.
5. Traditional TF-IDF demonstrates vector similarity but is not equivalent
   to modern neural embeddings.
6. Attention allows models to capture relationships between tokens.
7. Sampling configuration affects output variation.
8. LLM applications must consider latency, cost, privacy, and licensing.

### API limitation

No hosted LLM API experiments were performed because an OpenAI API key was
not configured in the local environment.

The notebook therefore uses reproducible local experiments where possible
and documents API-dependent concepts without fabricating measurements.